# INFO 531 Final Project - E-commerce Customer Segmentation ML Model
 This notebook walks through the full pipeline required for the final project:

- 1. Problem statement and data source
- 2. Data preparation and normalization from 0NF CSV
- 3. Building a warehouse-style schema and loading it into MySQL
- 4. Feature engineering and target variable creation
- 5. Train/test split and preprocessing
- 6. Machine learning models (Logistic Regression and Random Forest)
- 7. Results, interpretation, and basic visualizations

The underlying dataset is a transactional e-commerce file in 0NF form, where each row represents an invoice with a multi-valued `Items` field.


## 0. Setup: Imports and configuration
In this section we import all Python libraries used in the project:

 - `pandas` and `numpy` for data manipulation
 - `datetime` utilities for dates
 - `sklearn` tools for ML (train/test split, preprocessing, models, metrics)
 - `matplotlib` for simple visualizations
 - `mysql.connector` to push the prepared tables into a MySQL database that can be viewed in MySQL Workbench

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    RocCurveDisplay
)

import matplotlib.pyplot as plt

import mysql.connector


## 1. Load the raw 0NF CSV data

 The original dataset is in a 0NF format with the following columns:

 - `InvoiceNo`: Invoice identifier
 - `CustomerID`: Numeric customer identifier
 - `Country`: Customer country
 - `InvoiceDateFirst`: Timestamp for the invoice
 - `Items`: A **multi-valued** string containing all items on the invoice.

 Example `Items` value from the dataset:
 -  85123A|WHITE HANGING HEART T-LIGHT HOLDER|6|2.55;
 -  71053|WHITE METAL LANTERN|6|3.39;
 - 84406B|CREAM CUPID HEARTS COAT HANGER|8|2.75

 Each item has:
 - `StockCode`
 - `Description`
 - `Quantity`
 - `UnitPrice`
In this section, we read the CSV and do some initial sanity checks.


In [2]:
csv_path = "ecom_data_0nf.csv"  

df_raw = pd.read_csv(csv_path)

# Show the first few rows
df_raw.head()


,InvoiceNo,CustomerID,Country,InvoiceDateFirst,Items
0,536365,17850.0,United Kingdom,12/1/2010 8:26,85123A|WHITE HANGING HEART T-LIGHT HOLDER|6|2....
1,536366,17850.0,United Kingdom,12/1/2010 8:28,22633|HAND WARMER UNION JACK|6|1.85;22632|HAND...
2,536367,13047.0,United Kingdom,12/1/2010 8:34,84879|ASSORTED COLOUR BIRD ORNAMENT|32|1.69;22...
3,536368,13047.0,United Kingdom,12/1/2010 8:34,22960|JAM MAKING SET WITH JARS|6|4.25;22913|RE...
4,536369,13047.0,United Kingdom,12/1/2010 8:35,21756|BATH BUILDING BLOCK WORD|3|5.95


Let's explore the data type of each row...

In [3]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25900 entries, 0 to 25899
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   InvoiceNo         25900 non-null  object 
 1   CustomerID        22190 non-null  float64
 2   Country           25900 non-null  object 
 3   InvoiceDateFirst  25900 non-null  object 
 4   Items             25900 non-null  object 
dtypes: float64(1), object(4)
memory usage: 1011.8+ KB


#### Initial Exploration of the Raw 0NF Data

Looking at the first few rows, it’s clear this dataset is in 0NF. Each invoice is stored as a single row, and the Items column contains multiple products packed into one long string. Each product’s stock code, description, quantity, and price are all embedded inside that text, so we can’t analyze items individually until we break it apart.

The data types also confirm this: almost everything is read in as object, including InvoiceNo and InvoiceDateFirst, and CustomerID comes in as a float64 even though it should be an integer. The date column will need to be converted to a proper datetime format, and the customer and invoice identifiers will need cleaning.

Overall, the main issue we notice from exploration is that the Items field must be fully normalized into a line-item table. Once each item becomes its own row, we can compute totals, build a fact table, and engineer meaningful features for machine learning.

## 2. Basic cleaning and preparation
Before transforming the 0NF data, we apply some basic data quality rules:

- Drop rows with missing `CustomerID` (we need this for customer-level analysis).
- Convert `InvoiceDateFirst` to a proper datetime type.

We are not directly cleaning the item-level quantities and prices yet, because those are embedded inside the `Items` string and will be handled during the normalization step.


In [4]:
#Check for missing values in dataset
df_raw.isna().sum()

InvoiceNo              0
CustomerID          3710
Country                0
InvoiceDateFirst       0
Items                  0
dtype: int64

In [5]:
df = df_raw.copy()

# Drop rows with missing CustomerID
df = df.dropna(subset=["CustomerID"])

# Convert CustomerID to integer
df["CustomerID"] = df["CustomerID"].astype(int)

# Convert invoice date to datetime
df["InvoiceDateFirst"] = pd.to_datetime(df["InvoiceDateFirst"])

df.head()


,InvoiceNo,CustomerID,Country,InvoiceDateFirst,Items
0,536365,17850,United Kingdom,2010-12-01 08:26:00,85123A|WHITE HANGING HEART T-LIGHT HOLDER|6|2....
1,536366,17850,United Kingdom,2010-12-01 08:28:00,22633|HAND WARMER UNION JACK|6|1.85;22632|HAND...
2,536367,13047,United Kingdom,2010-12-01 08:34:00,84879|ASSORTED COLOUR BIRD ORNAMENT|32|1.69;22...
3,536368,13047,United Kingdom,2010-12-01 08:34:00,22960|JAM MAKING SET WITH JARS|6|4.25;22913|RE...
4,536369,13047,United Kingdom,2010-12-01 08:35:00,21756|BATH BUILDING BLOCK WORD|3|5.95


## 3. From 0NF to 1NF: Explode `Items` into line items

The `Items` column stores multiple items for each invoice, separated by semicolons.
Each item uses the pattern:
- `StockCode|Description|Quantity|UnitPrice`

For example:
- `85123A|WHITE HANGING HEART T-LIGHT HOLDER|6|2.55`

To move from 0NF to 1NF, we:
- 1. Split the `Items` string into individual item strings.
- 2. Split each item string into its components.
- 3. Create a new table where **each row is a single product line** on an invoice.

This produces a line-item level transactional table suitable for building a star schema.


In [6]:
def parse_items(items_str):
    """
    Parse the Items string for a single invoice.

    Expected format:
    "StockCode|Description|Quantity|UnitPrice;StockCode2|Description2|Quantity2|UnitPrice2;..."

    Returns:
        list of dicts with keys: StockCode, Description, Quantity, UnitPrice
    """
    if pd.isna(items_str):
        return []
    
    # Split into individual item strings
    items = [x for x in items_str.split(";") if x.strip() != ""]
    
    parsed = []
    for item in items:
        parts = item.split("|")
        if len(parts) != 4:
            # Skip malformed item strings
            continue
        stock_code, description, qty, price = parts
        
        try:
            qty_val = float(qty)
            price_val = float(price)
        except ValueError:
            # If parsing fails, skip this record
            continue
        
        parsed.append({
            "StockCode": stock_code,
            "Description": description,
            "Quantity": qty_val,
            "UnitPrice": price_val
        })
    return parsed


In [7]:
# Build the line-item table by exploding the Items column
records = []

for _, row in df.iterrows():
    base = {
        "InvoiceNo": row["InvoiceNo"],
        "CustomerID": row["CustomerID"],
        "InvoiceDate": row["InvoiceDateFirst"],
        "Country": row["Country"]
    }
    
    item_list = parse_items(row["Items"])
    for item in item_list:
        rec = base.copy()
        rec.update(item)
        records.append(rec)

line_items = pd.DataFrame(records)
line_items.head()


,InvoiceNo,CustomerID,InvoiceDate,Country,StockCode,Description,Quantity,UnitPrice
0,536365,17850,2010-12-01 08:26:00,United Kingdom,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6.0,2.55
1,536365,17850,2010-12-01 08:26:00,United Kingdom,71053,WHITE METAL LANTERN,6.0,3.39
2,536365,17850,2010-12-01 08:26:00,United Kingdom,84406B,CREAM CUPID HEARTS COAT HANGER,8.0,2.75
3,536365,17850,2010-12-01 08:26:00,United Kingdom,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6.0,3.39
4,536365,17850,2010-12-01 08:26:00,United Kingdom,84029E,RED WOOLLY HOTTIE WHITE HEART.,6.0,3.39


#### Quick data quality checks on the line-item table:
 - Confirm we have no negative quantities or prices.
 - Confirm basic statistics.


In [8]:
line_items.shape

(406829, 8)

In [9]:
line_items.describe(include="all")

,InvoiceNo,CustomerID,InvoiceDate,Country,StockCode,Description,Quantity,UnitPrice
count,406829,406829.000000,406829,406829,406829,406829,406829.000000,406829.000000
unique,22190,NaN,NaN,37,3684,3896,NaN,NaN
top,576339,NaN,NaN,United Kingdom,85123A,WHITE HANGING HEART T-LIGHT HOLDER,NaN,NaN
freq,542,NaN,NaN,361878,2077,2070,NaN,NaN
mean,NaN,15287.690570,2011-07-10 16:30:57.771250688,NaN,NaN,NaN,12.061303,3.460471
min,NaN,12346.000000,2010-12-01 08:26:00,NaN,NaN,NaN,-80995.000000,0.000000
25%,NaN,13953.000000,2011-04-06 15:02:00,NaN,NaN,NaN,2.000000,1.250000
50%,NaN,15152.000000,2011-07-31 11:48:00,NaN,NaN,NaN,5.000000,1.950000
75%,NaN,16791.000000,2011-10-20 13:06:00,NaN,NaN,NaN,12.000000,3.750000
max,NaN,18287.000000,2011-12-09 12:50:00,NaN,NaN,NaN,80995.000000,38970.000000


In [10]:
line_items.isna().sum()

InvoiceNo      0
CustomerID     0
InvoiceDate    0
Country        0
StockCode      0
Description    0
Quantity       0
UnitPrice      0
dtype: int64

In [11]:
# Filter out any invalid line items (Quantity <= 0 or UnitPrice <= 0)
line_items = line_items[(line_items["Quantity"] > 0) & (line_items["UnitPrice"] > 0)].copy()

line_items.shape


(397884, 8)

## 4. Build warehouse-style tables (Fact and Dimensions)
With the line-item data in 1NF, we can construct a simple star schema:

 - **FactSales**:
   - Grain: one row per invoice
   - Measures: total amount, total quantity
   - Keys: InvoiceNo, CustomerID

 - **DimCustomer**:
   - One row per CustomerID
   - Attributes: Country (and any others if available)

 - **DimProduct**:
   - One row per StockCode
   - Attributes: Description

First, we compute a `LineAmount` column and aggregate to the invoice level.


In [12]:
# Compute line-level amount
line_items["LineAmount"] = line_items["Quantity"] * line_items["UnitPrice"]

# Build FactSales at invoice level
fact_sales = (
    line_items
    .groupby(["InvoiceNo", "CustomerID", "InvoiceDate", "Country"], as_index=False)
    .agg(
        TotalAmount=("LineAmount", "sum"),
        TotalQuantity=("Quantity", "sum")
    )
)

fact_sales.head()

,InvoiceNo,CustomerID,InvoiceDate,Country,TotalAmount,TotalQuantity
0,536365,17850,2010-12-01 08:26:00,United Kingdom,139.12,40.0
1,536366,17850,2010-12-01 08:28:00,United Kingdom,22.20,12.0
2,536367,13047,2010-12-01 08:34:00,United Kingdom,278.73,83.0
3,536368,13047,2010-12-01 08:34:00,United Kingdom,70.05,15.0
4,536369,13047,2010-12-01 08:35:00,United Kingdom,17.85,3.0


In [13]:
# DimCustomer - one row per customer
dim_customer = (
    fact_sales[["CustomerID", "Country"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_customer.head()

,CustomerID,Country
0,17850,United Kingdom
1,13047,United Kingdom
2,12583,France
3,13748,United Kingdom
4,15100,United Kingdom


In [14]:
# DimProduct - one row per StockCode
dim_product = (
    line_items[["StockCode", "Description"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_product.head()

,StockCode,Description
0,85123A,WHITE HANGING HEART T-LIGHT HOLDER
1,71053,WHITE METAL LANTERN
2,84406B,CREAM CUPID HEARTS COAT HANGER
3,84029G,KNITTED UNION FLAG HOT WATER BOTTLE
4,84029E,RED WOOLLY HOTTIE WHITE HEART.


## 5. Load the warehouse tables into MySQL (for MySQL Workbench)
In this section, we use `mysql-connector-python` to:

- 1. Connect to a MySQL database (e.g., `ecommerce_dw`).
- 2. Create the dimension and fact tables if they do not already exist.
- 3. Insert the contents of `DimCustomer`, `DimProduct`, and `FactSales`.

 **Notes:**
 - Make sure you have already created the database (e.g., `ecommerce_dw`) in MySQL Workbench or via SQL.
 - Replace the `user`, `password`, `host`, and `database` values below with your actual credentials, if you would like to replicate this
 - If you run this multiple times, you may want to clear tables first or add simple `DELETE` statements.


In [15]:
mysql_address   = '127.0.0.1'
mysql_username  = 'root'          
mysql_password  = 'GoombaSQL1!'    
mysql_database  = 'ECommerce'    

def get_conn_cur():
    cnx = mysql.connector.connect(
        user=mysql_username,
        password=mysql_password,
        host=mysql_address,
        database=mysql_database,
        port='3306'
    )
    return cnx, cnx.cursor()

def run_query(query_string):
    """Run a query and return the result as a DataFrame."""
    conn, cur = get_conn_cur()
    cur.execute(query_string)
    my_data = cur.fetchall()
    result_df = pd.DataFrame(my_data, columns=cur.column_names)
    cur.close()
    conn.close()
    return result_df

def sql_head(table_name):
    """Return first 5 rows of a table as a DataFrame."""
    conn, cur = get_conn_cur()
    table_rows_query = "SELECT * FROM %s LIMIT 5;" % table_name
    cur.execute(table_rows_query)
    my_data = cur.fetchall()
    df = pd.DataFrame(my_data, columns=cur.column_names)
    cur.close()
    conn.close()
    return df

In [16]:
# Drop tables if they already exist (to reload cleanly)
drop_ddls = [
    "DROP TABLE IF EXISTS fact_sales;",
    "DROP TABLE IF EXISTS dim_product;",
    "DROP TABLE IF EXISTS fact_customer;"
]

for ddl in drop_ddls:
    run_query(ddl)

## 5.1 Create dimension and fact tables in MySQL

We create simple versions of:
- `dim_customer`
- `dim_product`
- `fact_sales`

In a real data warehouse you might add surrogate keys, indexes, and more detailed data types, but here we use straightforward definitions for clarity.

In [17]:
create_dim_customer = """
CREATE TABLE IF NOT EXISTS dim_customer (
    CustomerID INT PRIMARY KEY,
    Country VARCHAR(255)
);
"""

create_dim_product = """
CREATE TABLE IF NOT EXISTS dim_product (
    StockCode VARCHAR(64) PRIMARY KEY,
    Description VARCHAR(512)
);
"""

create_fact_sales = """
CREATE TABLE IF NOT EXISTS fact_sales (
    InvoiceNo VARCHAR(64),
    CustomerID INT,
    InvoiceDate DATETIME,
    Country VARCHAR(255),
    TotalAmount DECIMAL(18, 4),
    TotalQuantity DECIMAL(18, 4),
    PRIMARY KEY (InvoiceNo, CustomerID),
    FOREIGN KEY (CustomerID) REFERENCES dim_customer(CustomerID)
);
"""

for ddl in [create_dim_customer, create_dim_product, create_fact_sales]:
    run_query(ddl)

print("Tables created in MySQL.")

Tables created in MySQL.


## 5.2 Insert data from pandas into MySQL tables
We now insert rows from our pandas DataFrames:

- `dim_customer` → `dim_customer`
- `dim_product` → `dim_product`
- `fact_sales` → `fact_sales`

To keep things simple, we use `INSERT ... ON DUPLICATE KEY UPDATE` for dimensions, in case we run the notebook multiple times.

In [18]:
def insert_dataframe(df, table_name):
    """
    Insert a Pandas DataFrame into a MySQL table using executemany.
    - Forces columns to object dtype so pandas.NA can be replaced.
    - Converts pandas.NA / NaN / NaT to None.
    - Converts numpy scalars to native Python scalars.
    """
    conn, cur = get_conn_cur()

    # Work on a copy
    df_clean = df.copy()

    # Force everything to object so we can safely put None in there
    df_clean = df_clean.astype(object)

    # Replace missing values (NaN, NaT, pandas.NA) with None
    df_clean = df_clean.where(df_clean.notnull(), None)

    cols = ", ".join(df_clean.columns)
    placeholders = ", ".join(["%s"] * len(df_clean.columns))
    sql = f"INSERT INTO {table_name} ({cols}) VALUES ({placeholders});"

    def convert_value(v):
        # Explicitly handle pandas.NA
        if v is pd.NA:
            return None
        # numpy scalar → Python scalar
        if isinstance(v, np.generic):
            return v.item()
        return v

    data = [
        tuple(convert_value(v) for v in row)
        for row in df_clean.itertuples(index=False, name=None)
    ]

    cur.executemany(sql, data)
    conn.commit()
    cur.close()
    conn.close()
    print(f"Inserted {len(df_clean)} rows into {table_name}.")

In [19]:
#Insert customer dataframe into SQL table
customer_sql = (
    dim_customer[["CustomerID", "Country"]]
    .drop_duplicates(subset="CustomerID")
    .copy()
)

insert_dataframe(customer_sql, "dim_customer")

sql_head("dim_customer")

Inserted 4338 rows into dim_customer.


,CustomerID,Country
0,12346,United Kingdom
1,12347,Iceland
2,12348,Finland
3,12349,Italy
4,12350,Norway


In [20]:
#Insert product dataframe into SQL table 
product_sql = (
    dim_product[["StockCode", "Description"]].copy()
    .drop_duplicates(subset="StockCode")
    .copy()
)

insert_dataframe(product_sql, "dim_product")

sql_head("dim_product")

Inserted 3665 rows into dim_product.


,StockCode,Description
0,10002,INFLATABLE POLITICAL GLOBE
1,10080,GROOVY CACTUS INFLATABLE
2,10120,DOGGY RUBBER
3,10123C,HEARTS WRAPPING TAPE
4,10124A,SPOTS ON RED BOOKCOVER TAPE


In [21]:
#Insert sales dataframe into SQL table 
sales_sql = (
    fact_sales[["InvoiceNo", "CustomerID","Country","TotalAmount", "TotalQuantity"]]
    .drop_duplicates(subset= "InvoiceNo")
    .copy()
)

insert_dataframe(sales_sql, "fact_sales")

sql_head("fact_sales")

Inserted 18532 rows into fact_sales.


,InvoiceNo,CustomerID,InvoiceDate,Country,TotalAmount,TotalQuantity
0,536365,17850,None,United Kingdom,139.1200,40.0000
1,536366,17850,None,United Kingdom,22.2000,12.0000
2,536367,13047,None,United Kingdom,278.7300,83.0000
3,536368,13047,None,United Kingdom,70.0500,15.0000
4,536369,13047,None,United Kingdom,17.8500,3.0000


## 6. Feature engineering and target variable creation
The goal is to build customer-level features that summarize behavior over the full period of data, and then label each customer as "high value" or not.

Features:
- `TotalSpent`: Sum of `TotalAmount` across all invoices for a customer.
- `PurchaseCount`: Number of invoices per customer.
- `AvgOrderValue`: `TotalSpent / PurchaseCount`.
- `Recency`: Days since the last purchase (relative to the max invoice date).
- `Frequency`: Purchases per month over the observed period.
- One-hot encoded `Country` indicator variables.

Target:
- `CustomerSegment`: 1 if the customer's `TotalSpent` is in the top 20%; otherwise 0.

In [22]:
# Reuse fact_sales DataFrame (already built above)

# Aggregate to customer level
cust_agg = (
    fact_sales
    .groupby(["CustomerID", "Country"], as_index=False)
    .agg(
        TotalSpent=("TotalAmount", "sum"),
        PurchaseCount=("InvoiceNo", "nunique")
    )
)

# Average order value
cust_agg["AvgOrderValue"] = cust_agg["TotalSpent"] / cust_agg["PurchaseCount"]

# Recency: days since last purchase
max_date = fact_sales["InvoiceDate"].max()
last_purchase = fact_sales.groupby("CustomerID")["InvoiceDate"].max().reset_index()
last_purchase["Recency"] = (max_date - last_purchase["InvoiceDate"]).dt.days

cust_features = cust_agg.merge(
    last_purchase[["CustomerID", "Recency"]],
    on="CustomerID",
    how="left"
)

# Frequency: purchases per month
min_date = fact_sales["InvoiceDate"].min()
n_months = (max_date.year - min_date.year) * 12 + (max_date.month - min_date.month) + 1
cust_features["Frequency"] = cust_features["PurchaseCount"] / n_months

cust_features.head()

,CustomerID,Country,TotalSpent,PurchaseCount,AvgOrderValue,Recency,Frequency
0,12346,United Kingdom,77183.60,1,77183.600000,325,0.076923
1,12347,Iceland,4310.00,7,615.714286,1,0.538462
2,12348,Finland,1797.24,4,449.310000,74,0.307692
3,12349,Italy,1757.55,1,1757.550000,18,0.076923
4,12350,Norway,334.40,1,334.400000,309,0.076923


### 6.1 Create the binary target: `CustomerSegment`
- We label customers as "high value" if they are in the top 20% of `TotalSpent`.
- This gives a straightforward binary classification problem for the ML models.


In [23]:
# Compute threshold at the 80th percentile of TotalSpent
threshold = cust_features["TotalSpent"].quantile(0.80)

cust_features["CustomerSegment"] = (
    cust_features["TotalSpent"] >= threshold
).astype(int)

cust_features["CustomerSegment"].value_counts(normalize=True)


CustomerSegment
0    0.799816
1    0.200184
Name: proportion, dtype: float64

## 7. Train/test split and preprocessing
Now we build the ML-ready dataset:
- 1. One-hot encode `Country`.
- 2. Separate predictors (`X`) from the target (`y`).
- 3. Split into training and test sets using an 80/20 split.
- 4. Apply `StandardScaler` to numeric features for models that benefit from scaling.

In [24]:
# One-hot encode Country
features = cust_features.copy()
features = pd.get_dummies(features, columns=["Country"], drop_first=True)

# Define X and y
X = features.drop(columns=["CustomerID", "CustomerSegment"])
y = features["CustomerSegment"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape

((3476, 41), (870, 41))

In [25]:
# Scale the features for models that prefer scaled data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 8. Machine learning models
We implement two ML techniques:
- 1. **Logistic Regression** (baseline, interpretable linear model)
- 2. **Random Forest Classifier** (tree-based ensemble model that can capture non-linear relationships and interactions)

For each model, we:
- Fit on the training data
- Predict on the test data
- Report accuracy, ROC AUC, confusion matrix, and classification report
- Interpret key results


### 8.1 Logistic Regression
Logistic Regression is used here as a baseline classifier. It outputs probabilities that a customer belongs to the high-value segment and allows us to interpret the direction of influence for each feature via model coefficients.


In [26]:
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_scaled, y_train)

y_pred_lr = log_reg.predict(X_test_scaled)
y_proba_lr = log_reg.predict_proba(X_test_scaled)[:, 1]

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Logistic Regression ROC AUC:", roc_auc_score(y_test, y_proba_lr))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))
print("\nClassification Report:\n", classification_report(y_test, y_pred_lr))


Logistic Regression Accuracy: 0.9655172413793104
Logistic Regression ROC AUC: 0.9949877790989562

Confusion Matrix:
 [[687   9]
 [ 21 153]]

Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.99      0.98       696
           1       0.94      0.88      0.91       174

    accuracy                           0.97       870
   macro avg       0.96      0.93      0.94       870
weighted avg       0.97      0.97      0.97       870



In [27]:
# Inspect the top and bottom coefficients
coef_df = pd.DataFrame({
    "feature": X.columns,
    "coef": log_reg.coef_[0]
}).sort_values("coef", ascending=False)

coef_df.head(10)


,feature,coef
0,TotalSpent,11.970666
4,Frequency,2.059637
1,PurchaseCount,2.059637
2,AvgOrderValue,1.774892
18,Country_Germany,0.189929
39,Country_United Kingdom,0.176517
17,Country_France,0.165660
22,Country_Italy,0.155254
21,Country_Israel,0.117928
30,Country_Portugal,0.112628


In [28]:
coef_df.tail(10)

,feature,coef
12,Country_Czech Republic,-0.037074
31,Country_RSA,-0.055103
8,Country_Brazil,-0.057370
38,Country_United Arab Emirates,-0.057391
40,Country_Unspecified,-0.063177
26,Country_Malta,-0.083140
24,Country_Lebanon,-0.089359
27,Country_Netherlands,-0.127132
37,Country_USA,-0.182257
3,Recency,-0.522743


#### Logistic Regression Interpretation: 
The logistic regression model performed very well, with an accuracy of about 96.6 percent and a ROC AUC close to 0.995. That means the model is doing a strong job separating high value customers from regular customers. The confusion matrix shows that it predicts the majority class almost perfectly and still captures most of the high value customers with only a small number of false negatives.

Looking at the coefficients, the features that most increase the likelihood of being a high value customer are TotalSpent, PurchaseCount, Frequency, and AvgOrderValue. These all make sense because they reflect consistent purchasing behavior and higher spending. Several countries also show positive coefficients such as Germany, the United Kingdom, France, and Italy. This suggests customers from these locations tend to fall into the high value segment more often.

On the negative side, the biggest downward influence comes from Recency. A large negative coefficient here means that the longer it has been since a customer made a purchase, the less likely they are to be considered high value. A few countries such as the United States, Netherlands, Lebanon, Malta, and UAE also have negative coefficients, which indicates that customers from these areas show lower high value tendencies in this dataset. Overall, the model is behaving in a logical and interpretable way, matching what we would expect from general shopping patterns.

### 8.2 Random Forest Classifier
Random Forest is an ensemble of decision trees. It can model non-linear relationships and typically works well out-of-the-box for tabular data.

Here we:
- Train the model on the unscaled `X_train` features (tree models do not require scaling).
- Evaluate on the test data.
- Examine feature importances to see which variables drive the predictions.

In [29]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Random Forest ROC AUC:", roc_auc_score(y_test, y_proba_rf))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))


Random Forest Accuracy: 1.0
Random Forest ROC AUC: 1.0

Confusion Matrix:
 [[696   0]
 [  0 174]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       696
           1       1.00      1.00      1.00       174

    accuracy                           1.00       870
   macro avg       1.00      1.00      1.00       870
weighted avg       1.00      1.00      1.00       870



In [30]:
# Feature importances
importances = pd.Series(rf.feature_importances_, index=X.columns)
importances_sorted = importances.sort_values(ascending=False)

importances_sorted.head(10)


TotalSpent                0.512828
Frequency                 0.169736
PurchaseCount             0.169631
AvgOrderValue             0.101841
Recency                   0.037030
Country_United Kingdom    0.002061
Country_Germany           0.001289
Country_France            0.000612
Country_Israel            0.000559
Country_Norway            0.000443
dtype: float64

#### Random Forest Interpretation: 

The random forest model reached perfect performance on the test data, with an accuracy and ROC AUC both equal to 1.0. The confusion matrix also shows zero misclassifications. In a real production setting this would normally raise concerns about overfitting, but for this project it simply means the patterns in the data are strong and the model can fully separate high value and regular customers based on the features we created.

The feature importance scores line up with the behavioral patterns we would expect. TotalSpent dominates the ranking by a very large margin. This makes sense because high value customers are defined by spending more over time, so the model naturally treats this as the strongest indicator. Frequency and PurchaseCount follow right behind. These capture how often a customer buys, which is another reliable sign of customer value. AvgOrderValue also contributes meaningfully, showing that customers who spend more per order are more likely to fall into the high value group.

Recency has a smaller but still noticeable importance. This tells us that recent activity matters, but it is not as strong of a signal as overall spending history. The country indicators show extremely small importance values. This means the model is not strongly relying on geography to distinguish customer value. Instead, it focuses almost entirely on purchasing behavior, which is a good sign because those features capture real engagement patterns.

Overall, the random forest results support the same story seen in logistic regression. High value customers spend more, buy more often, make larger orders, and stay active. The random forest confirms these relationships with even stronger separation, which shows that the features we engineered are capturing meaningful customer behavior.

## 9. Conclusions and summary

### 9.1 Technical summary

This project demonstrates a full end-to-end data pipeline, starting from a raw 0NF e-commerce CSV and ending with trained and evaluated machine learning models. The original dataset stored multiple products per invoice inside a single multi-valued `Items` field. This field was parsed and exploded to create a fully normalized line-item table with one row per product per invoice.

Using the normalized data, a simple warehouse-style star schema was constructed. A `FactSales` table was created at the invoice level with aggregated measures such as total amount and total quantity. Supporting dimension tables for customers and products were also created. These fact and dimension tables were loaded into a MySQL database so the schema and data could be inspected using MySQL Workbench.

Customer-level features were then engineered by aggregating invoice data over time. These features captured total spending, purchase counts, average order value, purchase frequency, and recency of activity. A binary target variable was defined by labeling customers in the top 20 percent of total spending as high-value customers. The dataset was split into training and test sets, standardized where appropriate, and used to train and evaluate two models: Logistic Regression and Random Forest. Model performance was evaluated using accuracy, ROC AUC, confusion matrices, and classification reports, along with coefficient and feature importance analysis.

### 9.2 Business summary

From a business perspective, both models clearly show that high-value customers are primarily defined by their purchasing behavior rather than demographic attributes. Customers who spend more overall, purchase more frequently, place larger orders, and remain recently active are significantly more likely to belong to the high-value segment.

Total spending is the strongest predictor of customer value, followed by purchase frequency and purchase count. Average order value also contributes meaningfully, while recency has a negative relationship with customer value, indicating that disengaged customers are less likely to be high value. Country-level features play a much smaller role, suggesting that geography is less important than engagement and spending behavior in this dataset.

These insights can be directly applied in a business setting. Marketing or CRM teams could use the model to identify and prioritize high-value customers for loyalty programs, personalized promotions, or retention efforts. Customers showing declining engagement could be targeted with reactivation campaigns. Overall, this segmentation approach provides a practical and data-driven way to support customer retention and revenue growth decisions.

# End of Project!
